# Zero-Shot Autoregressive Counterfactual Generation with TabPFN

**Branch:** `zeroshot-tabpfn-cf`  ·  **Date:** 2026-06-15  ·  **Model:** TabPFN v2 (local, fully offline)

This notebook visualizes the results of using a pre-trained **TabPFN v2** model as a
*conditional density estimator* to generate features autoregressively — **no retraining,
no architecture changes**. We rely on `tabpfn-extensions`' `TabPFNUnsupervisedModel.impute()`,
injecting class conditioning via the **Y-as-column trick** (append the target label as an
extra categorical column, fix `Y = target` at impute time, NaN-mask the actionable features).

Two experiments:
1. **Single-feature estimation** (sanity gate) — mask one feature, reconstruct it from the
   target-class context.
2. **Counterfactual generation** — freeze immutable features, mask actionable ones, condition
   on the target class, evaluate validity / LOF plausibility / sparsity / actionability / L2 proximity.

Datasets: **MOONS** (2 features, 2-D) and **HELOC** (23 features, 6 immutable / 17 actionable).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import image as mpimg

pd.set_option("display.float_format", lambda v: f"{v:,.4g}")
RESULTS = Path("results")
if not RESULTS.exists():  # allow running from repo root too
    RESULTS = Path("experiments/zeroshot_cf/results")
assert RESULTS.exists(), f"results dir not found: {RESULTS.resolve()}"
print("Loading results from:", RESULTS.resolve())

## 1. Experiment 1 — Single-Feature Reconstruction (Sanity Gate)

For each feature we mask it in the test points and reconstruct it from a same-class context,
comparing TabPFN's conditional estimate against two baselines: the **marginal mean** (ignores
conditioning) and a **Ridge** regressor (cheap conditional reference). "Beats marginal" is the
gate signal — it tells us the conditional density is actually informative.

In [ ]:
exp1_moons = pd.read_csv(RESULTS / "exp1_moons.csv")
exp1_heloc = pd.read_csv(RESULTS / "exp1_heloc.csv")

def exp1_summary(df, name):
    return {
        "dataset": name,
        "n_features": len(df),
        "beats_marginal": f"{int(df.beats_marginal.sum())}/{len(df)} "
                          f"({df.beats_marginal.mean():.0%})",
        "mse_marginal": df.mse_marginal.mean(),
        "mse_tabpfn": df.mse_tabpfn.mean(),
        "mse_ridge": df.mse_ridge.mean(),
        "calib_10_90": df.calibration_10_90.mean(),
    }

summary = pd.DataFrame([exp1_summary(exp1_moons, "MOONS"),
                        exp1_summary(exp1_heloc, "HELOC")]).set_index("dataset")
summary

### HELOC: per-feature TabPFN vs. marginal-mean MSE

Bars below the diagonal/marginal line are features where conditioning helps. Green = TabPFN
beats the marginal baseline, red = it doesn't (typically sparse, heavy-tailed near-binary fields).

In [ ]:
df = exp1_heloc.sort_values("mse_marginal", ascending=False).copy()
colors = np.where(df.beats_marginal, "#2ca02c", "#d62728")
fig, ax = plt.subplots(figsize=(10, 7))
y = np.arange(len(df))
ax.barh(y - 0.2, df.mse_marginal, height=0.4, color="#9999bb", label="marginal mean")
ax.barh(y + 0.2, df.mse_tabpfn, height=0.4, color=colors, label="TabPFN (green=beats)")
ax.set_yticks(y); ax.set_yticklabels(df.feature_name, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("MSE (scaled space, lower is better)")
ax.set_title("HELOC single-feature reconstruction: TabPFN vs. marginal mean")
ax.legend()
plt.tight_layout(); plt.show()

n_beat = int(exp1_heloc.beats_marginal.sum())
print(f"TabPFN beats the marginal baseline on {n_beat}/{len(exp1_heloc)} HELOC features "
      f"({n_beat/len(exp1_heloc):.0%}). Gate verdict: PASS")

### MOONS: conditional spread vs. ground truth

The pre-generated scatter (true vs. reconstructed feature values) from the experiment runner:

In [ ]:
png = RESULTS / "exp1_moons.png"
if png.exists():
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(mpimg.imread(png)); ax.axis("off")
    plt.show()
else:
    print("exp1_moons.png not found; showing table instead")
    display(exp1_moons)

## 2. Experiment 2 — Counterfactual Generation

Freeze immutable features, NaN-mask actionable ones, fix `Y = target`, impute, evaluate.

**Metric directions:** validity ↑ (class flipped), LOF ↓ (more plausible / closer to training data),
sparsity ↓ (fewer features changed), **true_actionability** = 1.0 by construction (immutables frozen),
proximity L2 ↓, `frac_oob` ↓ (fraction of generated values outside the training [0,1] range).

In [ ]:
m_moons = pd.read_csv(RESULTS / "exp2_moons_metrics.csv")
m_heloc = pd.read_csv(RESULTS / "exp2_heloc_metrics.csv")
exp2 = pd.concat([m_moons, m_heloc], ignore_index=True).set_index("dataset")
cols = ["validity", "lof_scores_cf", "sparsity", "true_actionability",
        "proximity_l2_jaccard", "frac_oob"]
exp2[cols]

### Validity (with targets) and the HELOC plausibility collapse

Validity targets were **MOONS ≥ 0.70** and **HELOC ≥ 0.50** — both met. But the HELOC LOF score
is astronomically high and 72% of generated values fall out of range: masking 17/23 features leaves
only 6 immutables + Y to condition on, so the model extrapolates far outside the training manifold.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
ds = exp2.index.tolist()
x = np.arange(len(ds))

# validity vs target
targets = {"moons": 0.70, "heloc": 0.50}
axes[0].bar(x, exp2.validity, color=["#1f77b4", "#ff7f0e"])
for i, d in enumerate(ds):
    axes[0].hlines(targets[d], i - 0.4, i + 0.4, color="k", ls="--")
    axes[0].text(i, exp2.validity.iloc[i] + 0.02, f"{exp2.validity.iloc[i]:.2f}", ha="center")
axes[0].set_xticks(x); axes[0].set_xticklabels(ds); axes[0].set_ylim(0, 1)
axes[0].set_title("Validity (dashed = target)"); axes[0].set_ylabel("class-flip rate")

# LOF (log scale)
axes[1].bar(x, exp2.lof_scores_cf, color=["#1f77b4", "#ff7f0e"])
axes[1].set_yscale("log"); axes[1].set_xticks(x); axes[1].set_xticklabels(ds)
axes[1].set_title("LOF plausibility (log; lower=better)")
axes[1].axhline(1.0, color="green", ls=":", label="≈ training data")
axes[1].legend()

# OOB
axes[2].bar(x, exp2.frac_oob, color=["#1f77b4", "#ff7f0e"])
for i in range(len(ds)):
    axes[2].text(i, exp2.frac_oob.iloc[i] + 0.01, f"{exp2.frac_oob.iloc[i]:.0%}", ha="center")
axes[2].set_xticks(x); axes[2].set_xticklabels(ds); axes[2].set_ylim(0, 1)
axes[2].set_title("Out-of-range fraction (lower=better)")
plt.tight_layout(); plt.show()

### Concrete counterfactual examples

A few HELOC factual → counterfactual pairs (original feature space) generated by the runner:

In [ ]:
ex = RESULTS / "exp2_examples.md"
from IPython.display import Markdown
display(Markdown(ex.read_text() if ex.exists() else "_exp2_examples.md not found_"))

## 3. Refinement Sweep (Inference-Only)

We sweep **temperature**, **n_permutations**, and **context strategy** — no retraining.
Key question: can inference tuning fix HELOC's extrapolation problem?

In [ ]:
sw_moons = pd.read_csv(RESULTS / "exp2_sweep_moons.csv")
sw_heloc = pd.read_csv(RESULTS / "exp2_sweep_heloc.csv")
show = ["config_id", "temperature", "n_permutations", "context_type",
        "validity", "lof_scores_cf", "frac_oob", "proximity_l2_jaccard"]
print("MOONS sweep:")
display(sw_moons[show].sort_values("proximity_l2_jaccard"))
print("HELOC sweep:")
display(sw_heloc[show].sort_values("frac_oob"))

### MOONS: temperature is not a critical lever; HELOC: temperature cannot fix OOB

Note the counter-intuitive HELOC result — the near-MAP setting (`t=1e-9`) gives the **worst**
OOB (100%): under sparse conditioning even the modal prediction lands outside the training range.
This confirms the bottleneck is structural (information deficit from masking 17/23 features),
not a tunable parameter.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# MOONS: proximity vs validity, colored by context
for ctx, g in sw_moons.groupby("context_type"):
    axes[0].scatter(g.proximity_l2_jaccard, g.validity, s=90, label=ctx)
    for _, r in g.iterrows():
        axes[0].annotate(f"t={r.temperature:g}", (r.proximity_l2_jaccard, r.validity),
                         fontsize=7, xytext=(3, 3), textcoords="offset points")
axes[0].set_xlabel("proximity L2 (lower=better)"); axes[0].set_ylabel("validity")
axes[0].set_title("MOONS sweep: validity vs. proximity"); axes[0].legend(fontsize=8)

# HELOC: temperature vs OOB and validity
g = sw_heloc.sort_values("temperature")
ax = axes[1]; ax.plot(g.temperature, g.frac_oob, "o-", color="#d62728", label="frac OOB")
ax.plot(g.temperature, g.validity, "s-", color="#1f77b4", label="validity")
ax.set_xscale("log"); ax.set_xlabel("temperature (log)"); ax.set_ylim(0, 1.05)
ax.set_title("HELOC sweep: temperature can't fix OOB"); ax.legend()
plt.tight_layout(); plt.show()

## 4. Verdict

**Partially viable out-of-the-box.** *(Numbers updated post-review: Stage 7 fixed validity scoring
and RNG seeding; see `results/REPORT.md §Post-review corrections` for details.)*

| | MOONS (2-D) | HELOC (23-D) |
|---|---|---|
| Validity | **1.00** ✅ | **0.52** ✅ (barely) |
| LOF plausibility | **1.076** (≈ training data) | **~3.1B** ✗ |
| True actionability | **1.0** | **1.0** |
| OOB fraction | 0% | 72% |

**What works:** the Y-as-column conditioning mechanism is sound on low-dimensional, well-separated
data; immutability is a structural guarantee (frozen columns); single-feature reconstruction beats
the marginal baseline on 65% of HELOC features; everything runs fully offline.

**What doesn't (yet):** on HELOC, imputing 17/23 features from 7 observed values forces TabPFN to
extrapolate outside the training manifold — implausible CFs despite valid class flips. Temperature
does not help (MAP is *worse*). Proximity is poor (no minimum-change mechanism).

**Recommended next iteration:** (1) reduce the actionable set to 3–5 features; (2) k-NN context
selection by immutable features; (3) post-hoc projection of the CF onto the training manifold;
(4) feature-ordering DAG (most-predictable first). See `results/REPORT.md` for the full write-up.